In [ ]:
# CPU / BLAS thread controls
# Edit CPU_AFFINITY_RANGE and/or CPU_CAP before running the rest of the notebook.
import os
import multiprocessing as mp
from pathlib import Path

CPU_AFFINITY_RANGE = (90, 110)
CPU_CAP = 14

cpu_available = mp.cpu_count()
if os.environ.get("GENTRG_SMOKE_TEST") == "1":
    CPU_CAP = min(CPU_CAP, 4)

if CPU_AFFINITY_RANGE is not None:
    cpu_start, cpu_stop = [int(v) for v in CPU_AFFINITY_RANGE]
    if cpu_stop < cpu_start:
        raise ValueError(f"CPU_AFFINITY_RANGE must satisfy stop >= start, got {CPU_AFFINITY_RANGE}")
    selected_cpus = list(range(cpu_start, cpu_stop + 1))[: int(CPU_CAP)]
else:
    selected_cpus = None

cpu_use = len(selected_cpus) if selected_cpus is not None else max(1, min(int(CPU_CAP), cpu_available))
os.environ["MY_CPU_COUNT"] = str(cpu_use)
os.environ.setdefault("MPLCONFIGDIR", str(Path("/tmp") / "mplconfig_gaussian_entanglement_rg"))
for _var in (
    "OMP_NUM_THREADS",
    "OPENBLAS_NUM_THREADS",
    "MKL_NUM_THREADS",
    "VECLIB_MAXIMUM_THREADS",
    "NUMEXPR_NUM_THREADS",
    "NUMEXPR_MAX_THREADS",
):
    os.environ[_var] = str(cpu_use)

if selected_cpus is not None:
    try:
        os.sched_setaffinity(0, set(selected_cpus))
        print(f"CPU affinity set to cores [{min(selected_cpus)}, {max(selected_cpus)}]")
    except Exception as exc:
        print(f"CPU affinity not set: {exc}")

print(f"cpu_available={cpu_available}, cpu_use={cpu_use}, CPU_AFFINITY_RANGE={CPU_AFFINITY_RANGE}")


# Gaussian Entanglement RG for the OW Flattened Hamiltonian

This notebook constructs the half-filled ground state of the overcomplete-Wannier flattened Hamiltonian in the domain-wall benchmark geometry and applies a quasi-1D Gaussian entanglement RG along the periodic `y` direction. Each row is treated as a supersite with `q = 2*Nx` fermion orbitals.

The default run uses `Nx=20`, `Ny=40`, `nshell=1`, `alpha_1=1`, `alpha_2=30`, `DW=True`, and `dw_truncation=True`. Set `GENTRG_SMOKE_TEST=1` before execution for a reduced validation run.

## Setup

In [ ]:
import sys
import importlib
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib import font_manager


def find_repo_root(start):
    start = Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "src" / "fgtn" / "classA_U1FGTN.py").exists():
            return candidate
    raise FileNotFoundError(f"Could not find repo root from {start}")


ROOT = find_repo_root(Path.cwd())
NOTEBOOK_DIR = ROOT / "notebooks" / "flattened_hamiltonian_analysis" / "gaussian_entanglement_rg"
SMOKE_OUTPUT = os.environ.get("GENTRG_SMOKE_TEST") == "1"
DATA_DIR = NOTEBOOK_DIR / "data" / ("smoke" if SMOKE_OUTPUT else "")
FIGURES_DIR = NOTEBOOK_DIR / "figures" / ("smoke" if SMOKE_OUTPUT else "")
SRC = ROOT / "src"

for path in (DATA_DIR, FIGURES_DIR):
    path.mkdir(parents=True, exist_ok=True)

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

classA_U1FGTN_module = importlib.import_module("fgtn.classA_U1FGTN")
importlib.reload(classA_U1FGTN_module)
classA_U1FGTN = classA_U1FGTN_module.classA_U1FGTN

available_fonts = {f.name for f in font_manager.fontManager.ttflist}
if "CMU Sans Serif" in available_fonts:
    plt.rcParams["font.family"] = "CMU Sans Serif"
else:
    plt.rcParams["font.family"] = "DejaVu Sans"
    print("CMU Sans Serif not found; using DejaVu Sans")

plt.rcParams.update({
    "figure.dpi": 300,
    "savefig.dpi": 300,
    "axes.titlesize": 9,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 7,
    "lines.linewidth": 1.3,
    "mathtext.fontset": "cm",
})
np.set_printoptions(precision=6, suppress=True)

print(f"ROOT = {ROOT}")
print(f"NOTEBOOK_DIR = {NOTEBOOK_DIR}")
print(f"DATA_DIR = {DATA_DIR}")
print(f"FIGURES_DIR = {FIGURES_DIR}")


## Run Controls

In [ ]:
SMOKE_TEST = os.environ.get("GENTRG_SMOKE_TEST") == "1"

NX = 20
NY = 40
NSHELL = 1
ALPHA_1 = 1.0
ALPHA_2 = 30.0
DW = True
DW_TRUNCATION = True
TRIAL_ORBITAL = "X"

BASE_FIT_AY_MIN = 8
MIN_COARSE_NY = 10
ENTROPY_CLIP_EPS = 1e-12
DOMAIN_WALL_PROFILE_WIDTH = 1

if SMOKE_TEST:
    NX = 6
    NY = 16
    BASE_FIT_AY_MIN = 2
    MIN_COARSE_NY = 8

Q_ORBITALS = 2 * NX
HALF_FILLING = Q_ORBITALS * NY // 2

print(f"SMOKE_TEST = {SMOKE_TEST}")
print(f"Nx, Ny = {NX}, {NY}")
print(f"q = 2*Nx = {Q_ORBITALS}")
print(f"half filling = {HALF_FILLING} / {Q_ORBITALS * NY}")
print(f"nshell = {NSHELL}, alpha_1 = {ALPHA_1}, alpha_2 = {ALPHA_2}")
print(f"DW = {DW}, dw_truncation = {DW_TRUNCATION}, trial_orbital = {TRIAL_ORBITAL}")
print(f"RG target sequence stops at Ny >= {MIN_COARSE_NY}")


## Gaussian Free-Fermion Helpers

In [ ]:
def build_flattened_hamiltonian(model, trial_orbital=TRIAL_ORBITAL, dw_truncation=DW_TRUNCATION):
    nlayer = 2 * model.Nx * model.Ny
    model.construct_OW_projectors(
        nshell=model.nshell,
        DW=model.DW,
        trial_orbitals=trial_orbital,
        dw_truncation=dw_truncation,
    )
    WA_p = model.WF_Ap.reshape(nlayer, -1)
    WB_p = model.WF_Bp.reshape(nlayer, -1)
    WA_m = model.WF_Am.reshape(nlayer, -1)
    WB_m = model.WF_Bm.reshape(nlayer, -1)
    H_flat = WA_p @ WA_p.conj().T + WB_p @ WB_p.conj().T - WA_m @ WA_m.conj().T - WB_m @ WB_m.conj().T
    return 0.5 * (H_flat + H_flat.conj().T)


def build_half_filled_flattened_state(nx=NX, ny=NY):
    model = classA_U1FGTN(nx, ny, nshell=NSHELL, DW=DW, alpha_1=ALPHA_1, alpha_2=ALPHA_2)
    H_flat = build_flattened_hamiltonian(model)
    evals, evecs = np.linalg.eigh(H_flat)
    nlayer = H_flat.shape[0]
    n_occ = nlayer // 2

    occ_mask = evals < 0.0
    neg_count = int(np.count_nonzero(occ_mask))
    fallback_used = False
    if neg_count != n_occ:
        occ_mask = np.zeros_like(evals, dtype=bool)
        occ_mask[np.argsort(evals)[:n_occ]] = True
        fallback_used = True

    U_occ = evecs[:, occ_mask]
    C = U_occ @ U_occ.conj().T
    C = 0.5 * (C + C.conj().T)
    C_pm1 = 2.0 * C.conj() - np.eye(nlayer, dtype=np.complex128)

    return {
        "model": model,
        "H_flat": H_flat,
        "evals": np.real_if_close(evals),
        "C": C,
        "C_pm1": C_pm1,
        "n_occ": int(n_occ),
        "neg_count": neg_count,
        "fallback_used": bool(fallback_used),
        "spectral_gap": float(np.min(np.abs(evals))),
        "mean_abs_eval": float(np.mean(np.abs(evals))),
    }


def interval_indices(n_sites, q, length, start=0):
    n_sites = int(n_sites)
    q = int(q)
    length = int(length)
    start = int(start) % n_sites
    if length < 0 or length > n_sites:
        raise ValueError(f"length must lie in [0, {n_sites}], got {length}")
    rows = (np.arange(length, dtype=int) + start) % n_sites
    idx = np.concatenate([np.arange(row * q, (row + 1) * q, dtype=int) for row in rows])
    return idx


def entropy_from_correlation_submatrix(C_sub, clip_eps=ENTROPY_CLIP_EPS):
    C_sub = np.asarray(C_sub, dtype=np.complex128)
    C_sub = 0.5 * (C_sub + C_sub.conj().T)
    lambdas = np.linalg.eigvalsh(C_sub)
    lambdas = np.clip(np.real_if_close(lambdas), clip_eps, 1.0 - clip_eps)
    return float(-np.sum(lambdas * np.log(lambdas) + (1.0 - lambdas) * np.log(1.0 - lambdas)))


def entropy_for_interval(C, n_sites, q, length, start=0):
    if int(length) == 0:
        return 0.0
    idx = interval_indices(n_sites, q, length, start=start)
    return entropy_from_correlation_submatrix(C[np.ix_(idx, idx)])


def mean_entropy_profile(C, n_sites, q, max_length=None):
    n_sites = int(n_sites)
    if max_length is None:
        max_length = n_sites // 2
    lengths = np.arange(1, int(max_length) + 1, dtype=int)
    values = []
    records = []
    for length in lengths:
        entropies = [entropy_for_interval(C, n_sites, q, length, start=y0) for y0 in range(n_sites)]
        values.append(float(np.mean(entropies)))
        for y0, entropy in enumerate(entropies):
            records.append({"Ay": int(length), "y0": int(y0), "entropy": float(entropy)})
    return {
        "Ay_values": lengths,
        "avg_entropy": np.asarray(values, dtype=float),
        "records": records,
    }


def logchord_values(Ay_values, ny):
    Ay_values = np.asarray(Ay_values, dtype=float)
    return np.log((float(ny) / np.pi) * np.sin(np.pi * Ay_values / float(ny)))


def fit_ay_min_for_ny(ny):
    ny = int(ny)
    if ny >= 32:
        return int(BASE_FIT_AY_MIN)
    return max(2, int(ny // 5))


def fit_entropy_logchord(Ay_values, entropy_values, ny, fit_ay_min=None):
    Ay_values = np.asarray(Ay_values, dtype=int)
    entropy_values = np.asarray(entropy_values, dtype=float)
    if fit_ay_min is None:
        fit_ay_min = fit_ay_min_for_ny(ny)
    x = logchord_values(Ay_values, ny)
    fit_mask = (
        (Ay_values >= int(fit_ay_min))
        & (Ay_values <= int(ny // 2))
        & np.isfinite(x)
        & np.isfinite(entropy_values)
    )
    if int(np.count_nonzero(fit_mask)) < 3:
        raise ValueError(f"Need at least 3 fit points for Ny={ny}, fit_ay_min={fit_ay_min}")
    x_fit = x[fit_mask]
    y_fit = entropy_values[fit_mask]
    slope, intercept = np.polyfit(x_fit, y_fit, 1)
    y_hat = slope * x_fit + intercept
    ss_res = float(np.sum((y_fit - y_hat) ** 2))
    ss_tot = float(np.sum((y_fit - np.mean(y_fit)) ** 2))
    x_centered = x_fit - np.mean(x_fit)
    sxx = float(np.sum(x_centered ** 2))
    sigma2 = ss_res / float(len(x_fit) - 2) if len(x_fit) > 2 else np.nan
    slope_err = float(np.sqrt(max(sigma2, 0.0) / sxx)) if sxx > 0 and np.isfinite(sigma2) else np.nan
    r2 = 1.0 if np.isclose(ss_tot, 0.0) and np.isclose(ss_res, 0.0) else 1.0 - ss_res / ss_tot
    return {
        "slope": float(slope),
        "slope_err": float(slope_err),
        "intercept": float(intercept),
        "r2": float(r2),
        "c_est": float(3.0 * slope),
        "c_est_err": float(3.0 * slope_err) if np.isfinite(slope_err) else np.nan,
        "log_chord": x,
        "fit_mask": fit_mask,
        "fit_line": slope * x + intercept,
        "fit_ay_min": int(fit_ay_min),
    }


def entanglement_spectrum(C, n_sites, q, length=None, start=0):
    if length is None:
        length = n_sites // 2
    idx = interval_indices(n_sites, q, int(length), start=start)
    C_sub = C[np.ix_(idx, idx)]
    C_sub = 0.5 * (C_sub + C_sub.conj().T)
    return np.linalg.eigvalsh(C_sub)


## Quasi-1D Block Natural-Orbital RG

In [ ]:
def original_x_profile_from_embedding(embedding, nx, ny0, q_current):
    embedding = np.asarray(embedding, dtype=np.complex128)
    weights_by_original_mode = np.sum(np.abs(embedding) ** 2, axis=1)
    profile = np.zeros(int(nx), dtype=float)
    for y in range(int(ny0)):
        for x in range(int(nx)):
            base = 2 * int(nx) * y + 2 * x
            profile[x] += float(weights_by_original_mode[base] + weights_by_original_mode[base + 1])
    total = float(np.sum(profile))
    if total > 0.0:
        profile /= total
    return profile


def domain_wall_x_mask(nx, dw_locs, width=DOMAIN_WALL_PROFILE_WIDTH):
    mask = np.zeros(int(nx), dtype=bool)
    for loc in dw_locs:
        loc = int(loc) % int(nx)
        for delta in range(-int(width), int(width) + 1):
            mask[(loc + delta) % int(nx)] = True
    return mask


def block_natural_orbital_rg_step(C, n_sites, q, keep_per_block=None):
    C = np.asarray(C, dtype=np.complex128)
    n_sites = int(n_sites)
    q = int(q)
    if n_sites % 2 != 0:
        raise ValueError(f"n_sites must be even for binary RG, got {n_sites}")
    if keep_per_block is None:
        keep_per_block = q
    keep_per_block = int(keep_per_block)
    n_blocks = n_sites // 2
    fine_dim = n_sites * q
    coarse_dim = n_blocks * keep_per_block
    W = np.zeros((fine_dim, coarse_dim), dtype=np.complex128)
    rows = []

    for block in range(n_blocks):
        fine_start = 2 * block * q
        idx = np.arange(fine_start, fine_start + 2 * q, dtype=int)
        C_B = C[np.ix_(idx, idx)]
        C_B = 0.5 * (C_B + C_B.conj().T)
        occ, vecs = np.linalg.eigh(C_B)
        occ = np.clip(np.real_if_close(occ), 0.0, 1.0)
        chi = occ * (1.0 - occ)
        keep = np.argsort(chi)[-keep_per_block:]
        discard = np.setdiff1d(np.arange(2 * q), keep, assume_unique=False)
        # Preserve an occupation-sorted basis inside the retained subspace for stable plots.
        keep = keep[np.argsort(occ[keep])]
        W_block = vecs[:, keep]
        coarse_start = block * keep_per_block
        W[np.ix_(idx, np.arange(coarse_start, coarse_start + keep_per_block, dtype=int))] = W_block

        rows.append({
            "block": int(block),
            "kept_occ": occ[keep].copy(),
            "discarded_occ": occ[discard].copy(),
            "kept_chi": chi[keep].copy(),
            "discarded_chi": chi[discard].copy(),
            "mean_kept_chi": float(np.mean(chi[keep])),
            "mean_discarded_chi": float(np.mean(chi[discard])),
            "min_kept_chi": float(np.min(chi[keep])),
            "max_discarded_chi": float(np.max(chi[discard])),
        })

    C_next = W.conj().T @ C @ W
    C_next = 0.5 * (C_next + C_next.conj().T)
    return C_next, W, rows


def analyze_layer(C, embedding, layer, n_sites, q, nx, ny0, dw_locs):
    entropy_data = mean_entropy_profile(C, n_sites, q)
    fit = fit_entropy_logchord(entropy_data["Ay_values"], entropy_data["avg_entropy"], n_sites)
    x_profile = original_x_profile_from_embedding(embedding, nx, ny0, q)
    dw_mask = domain_wall_x_mask(nx, dw_locs)
    spec = entanglement_spectrum(C, n_sites, q)
    return {
        "layer": int(layer),
        "n_sites": int(n_sites),
        "q": int(q),
        "C": C,
        "embedding": embedding,
        "entropy_data": entropy_data,
        "fit": fit,
        "x_profile": x_profile,
        "dw_weight": float(np.sum(x_profile[dw_mask])),
        "entanglement_spectrum": spec,
    }


def run_gaussian_entanglement_rg(C0, nx, ny0, q, dw_locs, min_coarse_ny=MIN_COARSE_NY):
    layers = []
    rg_steps = []
    C = np.asarray(C0, dtype=np.complex128)
    n_sites = int(ny0)
    embedding = np.eye(C.shape[0], dtype=np.complex128)
    layer = 0
    layers.append(analyze_layer(C, embedding, layer, n_sites, q, nx, ny0, dw_locs))

    while n_sites > int(min_coarse_ny):
        C, W_step, block_rows = block_natural_orbital_rg_step(C, n_sites, q, keep_per_block=q)
        embedding = embedding @ W_step
        layer += 1
        n_sites //= 2
        rg_steps.append({"layer_to": int(layer), "W_step": W_step, "block_rows": block_rows})
        layers.append(analyze_layer(C, embedding, layer, n_sites, q, nx, ny0, dw_locs))

    return layers, rg_steps


## Build the Flattened-H Ground State

In [ ]:
state = build_half_filled_flattened_state(NX, NY)
model = state["model"]
C0 = state["C"]
DW_LOCS = list(getattr(model, "DW_loc", []))

print(f"DW locations = {DW_LOCS}")
print(f"H_flat shape = {state['H_flat'].shape}")
print(f"negative eigenvalue count = {state['neg_count']}")
print(f"filled modes = {state['n_occ']}")
print(f"fallback_used = {state['fallback_used']}")
print(f"spectral_gap = {state['spectral_gap']:.6e}")
print(f"mean_abs_eval = {state['mean_abs_eval']:.6e}")
print(f"projector idempotency ||C^2-C||_F = {np.linalg.norm(C0 @ C0 - C0):.6e}")


## Run the RG

In [ ]:
layers, rg_steps = run_gaussian_entanglement_rg(
    C0,
    nx=NX,
    ny0=NY,
    q=Q_ORBITALS,
    dw_locs=DW_LOCS,
    min_coarse_ny=MIN_COARSE_NY,
)

summary_rows = []
for item in layers:
    summary_rows.append({
        "smoke_test": bool(SMOKE_TEST),
        "Nx": int(NX),
        "Ny_initial": int(NY),
        "nshell": NSHELL,
        "alpha_1": float(ALPHA_1),
        "alpha_2": float(ALPHA_2),
        "dw_truncation": bool(DW_TRUNCATION),
        "layer": item["layer"],
        "Ny_current": item["n_sites"],
        "q_orbitals": item["q"],
        "fit_ay_min": item["fit"]["fit_ay_min"],
        "slope": item["fit"]["slope"],
        "slope_err": item["fit"]["slope_err"],
        "c_est": item["fit"]["c_est"],
        "c_est_err": item["fit"]["c_est_err"],
        "r2": item["fit"]["r2"],
        "domain_wall_x_weight_width1": item["dw_weight"],
    })

summary_df = pd.DataFrame(summary_rows)
summary_path = DATA_DIR / "gaussian_entanglement_rg_summary.csv"
summary_df.to_csv(summary_path, index=False)
print(f"Saved summary: {summary_path}")
summary_df


In [ ]:
occupation_rows = []
for step in rg_steps:
    layer_to = step["layer_to"]
    for block_row in step["block_rows"]:
        for mode, (occ, chi) in enumerate(zip(block_row["kept_occ"], block_row["kept_chi"])):
            occupation_rows.append({
                "layer_to": layer_to,
                "block": block_row["block"],
                "sector": "kept",
                "mode": mode,
                "occupation": float(occ),
                "chi": float(chi),
            })
        for mode, (occ, chi) in enumerate(zip(block_row["discarded_occ"], block_row["discarded_chi"])):
            occupation_rows.append({
                "layer_to": layer_to,
                "block": block_row["block"],
                "sector": "discarded",
                "mode": mode,
                "occupation": float(occ),
                "chi": float(chi),
            })
occupation_df = pd.DataFrame(occupation_rows)
occupation_path = DATA_DIR / "gaussian_entanglement_rg_occupations.csv"
occupation_df.to_csv(occupation_path, index=False)
print(f"Saved occupation spectra: {occupation_path}")
occupation_df.head()


In [ ]:
profile_rows = []
for item in layers:
    for x, weight in enumerate(item["x_profile"]):
        profile_rows.append({"layer": item["layer"], "x": int(x), "weight": float(weight)})
profile_df = pd.DataFrame(profile_rows)
profile_path = DATA_DIR / "gaussian_entanglement_rg_x_profiles.csv"
profile_df.to_csv(profile_path, index=False)
print(f"Saved x profiles: {profile_path}")
profile_df.head()


In [ ]:
spectrum_rows = []
for item in layers:
    eigs = np.asarray(item["entanglement_spectrum"], dtype=float)
    order = np.argsort(np.abs(eigs - 0.5))
    for rank, idx in enumerate(order[: min(80, len(order))]):
        spectrum_rows.append({
            "layer": item["layer"],
            "rank_near_half": int(rank),
            "lambda": float(eigs[idx]),
            "abs_lambda_minus_half": float(abs(eigs[idx] - 0.5)),
        })
spectrum_df = pd.DataFrame(spectrum_rows)
spectrum_path = DATA_DIR / "gaussian_entanglement_rg_entanglement_spectrum.csv"
spectrum_df.to_csv(spectrum_path, index=False)
print(f"Saved representative entanglement spectra: {spectrum_path}")
spectrum_df.head()


## Original Strip Entropy Fit

In [ ]:
item = layers[0]
entropy_data = item["entropy_data"]
fit = item["fit"]
fig, ax = plt.subplots(figsize=(3.375, 2.45))
ax.plot(fit["log_chord"], entropy_data["avg_entropy"], marker="o", markersize=3, linestyle="none", label="data")
ax.plot(fit["log_chord"], fit["fit_line"], color="black", label=rf"fit: $c_{{\rm est}}={fit['c_est']:.3f}$")
ax.set_xlabel(r"$\log\left[\frac{N_y}{\pi}\sin\left(\frac{\pi A_y}{N_y}\right)\right]$")
ax.set_ylabel(r"$\langle S(A_y)\rangle_{y_0}$")
ax.set_title(r"Original full-$x$ strip entropy")
ax.grid(alpha=0.25, linewidth=0.4)
ax.legend(frameon=False)
fig.tight_layout()
path = FIGURES_DIR / "original_strip_entropy_logchord_fit.png"
fig.savefig(path, dpi=300, bbox_inches="tight")
print(f"Saved figure: {path}")
plt.show()


## Central Charge Flow

In [ ]:
fig, ax = plt.subplots(figsize=(3.375, 2.35))
ax.errorbar(summary_df["layer"], summary_df["c_est"], yerr=summary_df["c_est_err"], marker="o", capsize=2)
ax.axhline(1.0, color="black", linestyle="--", linewidth=1.0, label=r"$c_{\rm eff}=1$")
for _, row in summary_df.iterrows():
    ax.annotate(f"Ny={int(row['Ny_current'])}", (row["layer"], row["c_est"]), textcoords="offset points", xytext=(4, 4), fontsize=7)
ax.set_xlabel("RG layer")
ax.set_ylabel(r"$c_{\rm est}=3m$")
ax.set_title(r"Log-chord central-charge estimate")
ax.grid(alpha=0.25, linewidth=0.4)
ax.legend(frameon=False)
fig.tight_layout()
path = FIGURES_DIR / "central_charge_flow.png"
fig.savefig(path, dpi=300, bbox_inches="tight")
print(f"Saved figure: {path}")
plt.show()


## Kept and Discarded Natural-Orbital Occupations

In [ ]:
fig, axes = plt.subplots(1, len(rg_steps), figsize=(max(3.375, 2.3 * len(rg_steps)), 2.45), sharey=True)
if len(rg_steps) == 1:
    axes = [axes]
for ax, step in zip(axes, rg_steps):
    layer_to = step["layer_to"]
    plot_df = occupation_df[occupation_df["layer_to"] == layer_to]
    for sector, color, alpha in [("discarded", "tab:gray", 0.45), ("kept", "tab:blue", 0.75)]:
        vals = plot_df[plot_df["sector"] == sector]["occupation"].to_numpy()
        vals = np.sort(vals)
        ax.scatter(np.arange(vals.size), vals, s=5, color=color, alpha=alpha, label=sector)
    ax.axhline(0.5, color="black", linestyle="--", linewidth=0.8)
    ax.set_title(rf"Layer {layer_to}")
    ax.set_xlabel("mode rank")
    ax.grid(alpha=0.2, linewidth=0.35)
axes[0].set_ylabel(r"natural occupation $n_a$")
axes[-1].legend(frameon=False, loc="best")
fig.tight_layout()
path = FIGURES_DIR / "natural_orbital_occupations.png"
fig.savefig(path, dpi=300, bbox_inches="tight")
print(f"Saved figure: {path}")
plt.show()


## Retained-Subspace Weight Versus Original `x`

In [ ]:
fig, ax = plt.subplots(figsize=(3.375, 2.45))
for item in layers:
    ax.plot(np.arange(NX), item["x_profile"], marker="o", markersize=2.5, label=rf"layer {item['layer']}, $N_y={item['n_sites']}$")
for dw in DW_LOCS:
    ax.axvline(dw, color="black", linestyle="--", linewidth=0.8, alpha=0.65)
ax.set_xlabel(r"original $x$")
ax.set_ylabel("retained-subspace weight")
ax.set_title(r"Retained modes in the original $x$ direction")
ax.grid(alpha=0.25, linewidth=0.4)
ax.legend(frameon=False, ncol=1)
fig.tight_layout()
path = FIGURES_DIR / "retained_x_profile.png"
fig.savefig(path, dpi=300, bbox_inches="tight")
print(f"Saved figure: {path}")
plt.show()


## Representative Entanglement Spectrum Flow

In [ ]:
fig, ax = plt.subplots(figsize=(3.375, 2.45))
for item in layers:
    eigs = np.asarray(item["entanglement_spectrum"], dtype=float)
    near = eigs[np.argsort(np.abs(eigs - 0.5))[:40]]
    near = np.sort(near)
    ax.scatter(np.full_like(near, item["layer"], dtype=float), near, s=8, alpha=0.65, label=rf"$N_y={item['n_sites']}$")
ax.axhline(0.5, color="black", linestyle="--", linewidth=0.8)
ax.set_xlabel("RG layer")
ax.set_ylabel(r"entanglement eigenvalue $\lambda$")
ax.set_title("Half-chain entanglement spectrum near $1/2$")
ax.grid(alpha=0.25, linewidth=0.4)
ax.legend(frameon=False, loc="best")
fig.tight_layout()
path = FIGURES_DIR / "entanglement_spectrum_flow.png"
fig.savefig(path, dpi=300, bbox_inches="tight")
print(f"Saved figure: {path}")
plt.show()


## Smoke-Test Interpretation

For the domain-wall benchmark, the full-$x$ strip should behave as a quasi-1D nonchiral Dirac theory built from the two counter-propagating chiral domain-wall channels, giving target `c_est = 1`. The natural-orbital RG is only a smoke test: it uses `U = I` and block-dependent isometries, so it tests local Gaussian mode separation before attempting an optimized scale-invariant MERA.